#### Exponential Time Differencing  
Created May 7, 2026

In this notebook we implement the Exponential Time-Differencing (ETD) method for solving ODE systems of the form 
#### 
\begin{align*}
    u_t = Au + f(u,t)
\end{align*}

where $A$ is an $N \times N$ constant matrix and $f:\mathbb{R}^{N} \times \mathbb{R} \to \mathbb{R}$ is some non-linear function. ETD methods are especially useful for ODEs of the above form, as the linear term $Au$ often makes the ODE system stiff; ETD handles this by "integrating the linear part exactly" (in a sense discussed below). More precisely, ETD makes use of the following *variation of parameters* formula: 
\begin{align*}
    \fbox{$u(t_{n+1}) = e^{\Delta t A} u_n + \int_{t_n}^{t_{n+1}} e^{(t_{n+1} - s)A} f(u(s),s)ds \,$}
\end{align*}

A fixed-step ETD scheme is just a scheme of the form 
\begin{align*}
   \qquad u_{n+1} = e^{\Delta t A} u_n + I_n, \qquad n = 0,1,2,\ldots
\end{align*}
where $I_n = I_n(u_n,\ldots,u_{n-k})$ is an approximation to the integral term. ETD Euler is obtained by making the approximation $f(u(\cdot),\cdot) \approx f(u(t_n), t_n))$ on $[t_{n},t_{n+1}]$: 
\begin{align*}
    I_n \approx \left(\int_{t_n}^{t_{n+1}} e^{(t_{n+1} - s)A}\right) f(u_n,t_n) = \left(\int_{0}^{\Delta t} e^{sA} \,ds \right) f(u_n,t_n). 
\end{align*}

The last integral can be expressed as a series: 
\begin{align*}
    \int_{0}^{\Delta t} e^{sA} \,ds = \int_{0}^{\Delta t} \sum_{k=0}^{\infty} \frac{(sA)^k}{k!} \,ds = \sum_{k=0}^{\infty} \int_{0}^{\Delta t}  \frac{(sA)^k}{k!} \,ds = \sum_{k=0}^{\infty} \frac{(\Delta t)^{k+1}A^k}{(k+1)!} = \Delta t \underbrace{\left(\sum_{k=0}^{\infty} \frac{(\Delta t A)^k}{(k+1)!}\right)}_{=:\varphi_1(\Delta t A)}
\end{align*}

The last series in the above line is the definition for the matrix-valued function $\varphi_1: \mathbb{R}^{n \times n} \to \mathbb{R}^{n \times n}$, which comes up frequently in ETD schemes (along with other so-called *phi-functions*). Note that if $z$ is a scalar, then 
\begin{align*}
   z \varphi_1(z) = \sum_{k=0}^{\infty} \frac{z^{k+1}}{(k+1)!} = \sum_{k=1}^{\infty} \frac{z^k}{k!} = e^z - 1
\end{align*}
and so 
\begin{align*}
   \varphi_1(z) = \frac{e^z - 1}{z} \quad \text{for } z \in \mathbb{R} \text{ or } \mathbb{C}. 
\end{align*}

Thus, the ETD Euler scheme in terms of $\varphi_1$ is: 
\begin{align*}
   \textbf{ETD Euler:} \quad \fbox{$\displaystyle u_{n+1} = u_n e^{\Delta t A} + \Delta t \, \varphi_1(\Delta t A) f(u_n,t_n)$}
\end{align*}

Or assuming that $A$ is invertible, 
\begin{align*}
    \fbox{$\displaystyle u_{n+1} = u_n e^{\Delta t A} + A^{-1} (e^{\Delta t A} - I) f(u_n,t_n)$}
\end{align*}

**Remarks:** 
- The matrix inverse $A^{-1}$ can be computed once at the outset. This shouldn't be a problem.  
- We have to compute the matrix exponentials $e^{\Delta t A}$ at every time step, which is expensive. Don't try using a Taylor series approximation;
- It's better to use some specialized tools for this, such as `ExponentialUtilities.jl`. 


<!--
In the ETD schemes we will make use of the following functions:
\begin{align}
 \varphi_1(z) = \frac{e^z - 1}{z}, \qquad \varphi_2(z) = \frac{e^z - 1 - z}{z^2}. 
\end{align}

The **ETD-Euler scheme**: 
\begin{align*}
adf
\end{align*}

We implement the so-called **ETD2RK** scheme from Cox and Matthews (2002): 
\begin{flalign*}
   a &= e^{L \Delta t} u^n + \Delta t \varphi_1(L \Delta t) f(u^n, t^n) &&  \\[4pt]
   u^{n+1} &= a + \Delta t \varphi_2(L \Delta t) \left[f(a,t^{n+1}) - f(u^n,t^n) \right]  && 
\end{flalign*}
-->

In [1]:
using OrdinaryDiffEq, LaTeXStrings, Printf, UnPack, NBInclude, LinearAlgebra, Random, FFTW, ExponentialUtilities
@nbinclude("phi_functions.ipynb")

In [2]:
function etd_euler(A, f, u0; tspan::Tuple, Δt::Float64, p = nothing, ϵ::Float64 = 1e-3)
    """
    Solves the ODE u_t = Au + f(u,p,t)  on [tspan[1], tspan[2]] using the ETD-Euler method where: 
    - u: [t0,tf] -> R^N is the ODE solution
    - A is an N x N constant matrix 
    - f is a nonlinear function from R^N × [t0,tf] to R^N
    - p an optional parameter for the function 'f' (must be included in the function signature though) 
    
    PARAMETERS
    ----------
    A :: N x N matrix (or a scalar) 
    f :: a function `f(u,p,t)` from R^N to R (the non-linear term) where u is the state, p is a scalar, and t is time. 
    u0 :: the initial condition; a scalar (if N = 1) or a vector (if N > 1) 
    Δt :: the time step 
    tspan :: time interval over which to integrate (a tuple) 
    p :: parameter tuple for the reaction term (if required) 
    
    RETURNS
    -------
    sol.u :: vector of solution iterates 
    sol.t :: time values at which the solution was computed 
    sol.Δt :: the time step 
    """

    t0, tf = tspan                                         #Endpoints of time interval
    num_steps = Int(floor((tf - t0)/Δt))                   #Number of steps to be taken by the ODE solver 
    num_save = num_steps + 1                               #Number of solution iterates to save (****INCLUDES THE INITIAL CONDITION, `u0`****) 
    t = collect(range(t0, step = Δt, length = num_save))   #Time values at which to record the solution 
                                                           #NOTE: t = [t_0, t_1,...,t_M], where t_i = t_0 + iΔt for i=0,1,...,M, where M := num_steps 
    
    u = [zero(u0) for _ in 1:num_save]            #Initialize array to store the solution u [Note: u = [u[1],...,u[M+1]]
    u[1] = u0                                    #Record the initial condition  

    E, Φ₁ = phis(Δt*A, 1; ϵ = ϵ)
      
    for j=1:num_steps     #j+1 = 2,3,...,num_save 
        u[j+1] = E * u[j] + Δt * (Φ₁ * f(u[j], p, t[j]))
    end 

    return (u = u, t = t, Δt = Δt) 
end 

etd_euler (generic function with 1 method)

In [33]:
#phiv(1.0, [1.0 2.0; 3.0 4.0], [1.0,2.0], 3)
 # #NEW VERSION (using `phiv` from ExponentialUtilities.jl)  [WAY SLOWER!]
    # for j=1:num_steps     #j+1 = 2,3,...,num_save 

    #     phi_u = phiv(Δt, A, u[j], 0)[:,1]                 #phi_u = ϕ₀(Δt*A) u[j]   (= exp(Δt*A) u[j])
    #     phi_f = phiv(Δt, A, f(u[j], p, t[j]), 1)[:,2]     #phi_f = ϕ₁(Δ
    #     u[j+1] = phi_u + Δt * phi_f 
    # end 